In [ ]:
!pip install -q timm albumentations safetensors wandb huggingface_hub datasets scikit-learn

In [ ]:
import subprocess

result = subprocess.run(
    ["git", "clone", "https://github.com/hssling/anemia-detection-ai.git"],
    capture_output=True,
    text=True,
)
print(result.stdout)
import sys  # noqa: E402

sys.path.insert(0, "/kaggle/working/anemia-detection-ai")

In [ ]:
import os

from huggingface_hub import login

login(token=os.environ["HF_TOKEN"])
import wandb  # noqa: E402

wandb.login(key=os.environ["WANDB_API_KEY"])

In [ ]:
from datasets import load_dataset

dataset = load_dataset("hssling/anemia-conjunctiva-nailbed")
train_conj = [r for r in dataset["train"] if r["site"] == "conjunctiva"]
val_conj = [r for r in dataset["val"] if r["site"] == "conjunctiva"]
train_nail = [r for r in dataset["train"] if r["site"] == "nailbed"]
val_nail = [r for r in dataset["val"] if r["site"] == "nailbed"]
print(f"Train conj: {len(train_conj)}, val conj: {len(val_conj)}")
print(f"Train nail: {len(train_nail)}, val nail: {len(val_nail)}")

In [ ]:
import pathlib

import yaml

config = yaml.safe_load(open("anemia-detection-ai/training/config.yaml"))
from training.cross_validation import run_cross_validation  # noqa: E402

cv_summary_conj = run_cross_validation(
    rows=train_conj + val_conj,
    model_name="efficientnet_b4",
    config=config,
    output_dir=pathlib.Path("/kaggle/working/outputs/cv_conj"),
)
print("CV conjunctiva:", cv_summary_conj)

In [ ]:
from training.train import train_model

final_metrics_conj = train_model(
    model_name="efficientnet_b4",
    train_rows=train_conj + val_conj,
    val_rows=val_conj,
    config=config,
    output_dir=pathlib.Path("/kaggle/working/outputs/final"),
    fold=99,
    run_name="efficientnet_b4_conjunctiva_final",
)

In [ ]:
cv_summary_nail = run_cross_validation(
    rows=train_nail + val_nail,
    model_name="efficientnet_b4",
    config=config,
    output_dir=pathlib.Path("/kaggle/working/outputs/cv_nail"),
)

In [ ]:
final_metrics_nail = train_model(
    model_name="efficientnet_b4",
    train_rows=train_nail + val_nail,
    val_rows=val_nail,
    config=config,
    output_dir=pathlib.Path("/kaggle/working/outputs/final"),
    fold=98,
    run_name="efficientnet_b4_nailbed_final",
)

In [ ]:
from training.models.ensemble import AnemiaEnsemble

w_conj, w_nail = AnemiaEnsemble.find_best_weights(
    conj_ckpt="/kaggle/working/outputs/final/efficientnet_b4_fold99_best.safetensors",
    nail_ckpt="/kaggle/working/outputs/final/efficientnet_b4_fold98_best.safetensors",
    val_rows_conj=val_conj,
    val_rows_nail=val_nail,
    config=config,
)
print(f"Best ensemble weights: w_conj={w_conj:.2f}, w_nail={w_nail:.2f}")

In [ ]:
from training.push_model_to_hf import push_all_models

push_all_models(
    conj_ckpt="/kaggle/working/outputs/final/efficientnet_b4_fold99_best.safetensors",
    nail_ckpt="/kaggle/working/outputs/final/efficientnet_b4_fold98_best.safetensors",
    cv_summary_conj=cv_summary_conj,
    cv_summary_nail=cv_summary_nail,
    w_conj=w_conj,
    w_nail=w_nail,
    config=config,
)
print("All models pushed to HuggingFace Hub")